In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


class HARRV:
    """
    HAR-RV model:
    RV_t = β0 + βd * RV_{t-1} + βw * RV_{t-1:t-5} + βm * RV_{t-1:t-22} + error

    Options:
        use_log = True : regress in log(RV)
    """

    def __init__(self, use_log=True):
        self.use_log = use_log
        self.model = None
        self.fitted = False
        self.columns = None  # will store regressors order

    # ---------------------------------------------------------
    # Load / compute realized variance
    # ---------------------------------------------------------
    @staticmethod
    def compute_realized_variance(df, price_col="price", datetime_col="datetime"):
        """
        Accepts intraday data with timestamps & prices.
        Returns a daily RV (variance) series.
        """
        df["date"] = pd.to_datetime(df[datetime_col]).dt.date
        df = df.sort_values("date").copy()
        df["logret"] = np.log(df[price_col]).diff()
        
        rv = df.groupby("date")["logret"].var()
        rv = rv.dropna()
        return rv.to_frame("rv")

    # ---------------------------------------------------------
    # Create HAR features
    # ---------------------------------------------------------
    @staticmethod
    def _make_har_features(rv_df):
        df = rv_df.copy()
        df["rv_lag1"] = df["rv"].shift(1)
        df["rv_week"] = df["rv"].rolling(5).mean().shift(1)
        df["rv_month"] = df["rv"].rolling(22).mean().shift(1)
        df = df.dropna()
        return df

    # ---------------------------------------------------------
    # Fit model
    # ---------------------------------------------------------
    def fit(self, rv_df):
        """
        Fit HAR-RV given a dataframe with 'rv' column.
        """
        df = self._make_har_features(rv_df)

        if self.use_log:
            df["y"] = np.log(df["rv"])
            X = np.log(df[["rv_lag1", "rv_week", "rv_month"]])
        else:
            df["y"] = df["rv"]
            X = df[["rv_lag1", "rv_week", "rv_month"]]

        X = sm.add_constant(X)

        self.model = sm.OLS(df["y"], X).fit()
        self.fitted = True
        self.columns = X.columns

        return self.model

    # ---------------------------------------------------------
    # Predict next-day RV
    # ---------------------------------------------------------
    def forecast_next(self, rv_df):
        """
        Forecast next day's RV using the last available row.
        """
        if not self.fitted:
            raise Exception("Model not fitted yet.")

        df = self._make_har_features(rv_df)
        last = df.iloc[-1:]

        if self.use_log:
            X = np.log(last[["rv_lag1", "rv_week", "rv_month"]])
        else:
            X = last[["rv_lag1", "rv_week", "rv_month"]]

        X = sm.add_constant(X)
        X = X[self.columns]  # reorder

        pred = self.model.predict(X).iloc[0]
        return float(np.exp(pred)) if self.use_log else float(pred)

    # ---------------------------------------------------------
    # Predict entire sample (in-sample or out-of-sample)
    # ---------------------------------------------------------
    def predict(self, rv_df):
        if not self.fitted:
            raise Exception("Model not fitted yet.")

        df = self._make_har_features(rv_df)

        if self.use_log:
            X = np.log(df[["rv_lag1", "rv_week", "rv_month"]])
        else:
            X = df[["rv_lag1", "rv_week", "rv_month"]]

        X = sm.add_constant(X)
        X = X[self.columns]

        preds = self.model.predict(X)
        return np.exp(preds) if self.use_log else preds

    # ---------------------------------------------------------
    # Train/test evaluation
    # ---------------------------------------------------------
    def evaluate(self, rv_df, split=0.8, plot=True):
        df = self._make_har_features(rv_df)
        n = len(df)
        s = int(n * split)

        train = df.iloc[:s]
        test = df.iloc[s:]

        # Fit on train
        self.fit(train)

        # Predict on test
        preds = self.predict(test)

        # Actuals
        actuals = test["rv"]

        mse = np.mean((preds - actuals) ** 2)
        mae = np.mean(np.abs(preds - actuals))

        if plot:
            plt.figure(figsize=(12, 5))
            plt.plot(actuals.index, actuals.values, label="Actual RV")
            plt.plot(actuals.index, preds, "--", label="Predicted RV")
            plt.title("HAR-RV Test-Set Forecast")
            plt.legend()
            plt.tight_layout()
            plt.show()

        return {"mse": float(mse), "mae": float(mae)}


# ==============================================================
# Example usage (comment out if using as a library)
# ==============================================================

if __name__ == "__main__":
    # ---------------------------------------------------------
    # EXAMPLE: simulate intraday data
    # ---------------------------------------------------------
    print("Generating synthetic intraday price data...")
    np.random.seed(0)

    dt = pd.date_range("2021-01-01", "2021-06-30", freq="5T")  # 5-min bars
    prices = 100 * np.exp(np.cumsum(0.0002 + 0.002 * np.random.randn(len(dt))))

    df_prices = pd.DataFrame({"datetime": dt, "price": prices})

    # Compute RV
    rv = HARRV.compute_realized_variance(df_prices)

    # Fit model
    model = HARRV(use_log=True)
    model.fit(rv)

    print(model.model.summary())

    # Forecast next day
    f = model.forecast_next(rv)
    print("Next day RV forecast:", f)

    # Evaluate
    metrics = model.evaluate(rv)
    print(metrics)



har = HARRV()
har.fit(rv_df)
har.forecast_next(rv_df)
har.predict(rv_df)
har.evaluate(rv_df)